## Step 1: Confirm the Bronze table

In [0]:
SELECT *
FROM workspace.michelin_analytics.bronze_michelin_restaurants
LIMIT 20;

Name,Address,Location,Price,Cuisine,Longitude,Latitude,PhoneNumber,Url,WebsiteUrl,Award,GreenStar,FacilitiesAndServices,Description
8 1/2 Otto e Mezzo - Bombana,"Shop 202, 2F, Alexandra House, 18 Chater Road, Central, Hong Kong, Hong Kong SAR China","Hong Kong, Hong Kong SAR China",$$$$,Italian,114.15867,22.2814635,85225378859,https://guide.michelin.com/en/hong-kong-region/hong-kong/restaurant/8%C2%BD-otto-e-mezzo-bombana,https://www.ottoemezzobombana.com,3 Stars,0,"Air conditioning,Car park,Interesting wine list,Valet parking,Wheelchair access","(The restaurant is temporarily closed.) Owner-chef Umberto Bombana’s restaurant, which oozes Italian charm and passion from every pore, is named after one of the chef’s favourite Fellini films. The classical Italian lineup stars first-class ingredients, like lamb from Aveyron, scallops from Hokkaido and Wagyu beef from Australia. The pasta course is always a highlight – scampi risotto and trenette with sea urchin are just two of the standouts. The delightful service makes everyone feel special."
ABaC,"Avenida del Tibidabo 1, Barcelona, 08022, Spain","Barcelona, Spain",$$$$,Creative,2.1365112,41.4104497,34933196600,https://guide.michelin.com/en/catalunya/barcelona/restaurant/abac,https://abacrestaurant.com,3 Stars,0,"Air conditioning,Interesting wine list,Terrace","Tradition, cutting-edge, flavour, attitude and passion, along with a consistent approach, define to perfection the cooking of Jordi Cruz. In his elegant restaurant boasting views of a tranquil garden dotted with some amazing works of contemporary art, he offers diners a unique gastronomic experience that constantly evolves in line with ingredients from the changing seasons. These are always combined with his technical ability as he revisits Catalan and other Mediterranean flavours with a nod to influences from elsewhere around the globe. The experience here, which begins with appetisers in the kitchen, is centred around a single tasting menu featuring new recipes alongside the restaurant’s renowned signature dishes. One of the most extraordinary dishes is the Aqua Mirabilis dessert, an original tribute to Johann Maria Farina, who, in 1709, founded the oldest perfume factory in the world. This course, which plays with different textures, allows you to taste a scent through a wonderful sensory game."
AM par Alexandre Mazzia,"9 rue François-Rocca, Marseille, 13008, France","Marseille, France",$$$$,Creative,5.3862325,43.2701104,33491248363,https://guide.michelin.com/en/provence-alpes-cote-dazur/marseille/restaurant/am-par-alexandre-mazzia,https://www.alexandre-mazzia.com/,3 Stars,0,"Air conditioning,Interesting wine list,Pets allowed everywhere","In his eatery near the Stade Vélodrome, this artist-cum-chef on a perpetual creative quest has elevated the small portion to the realm of art. He is a virtuoso when it comes to spices, roasting and smoking and his childhood in Congo flows freely through his cuisine. This distinctive hallmark is further enriched by a unique use of vinegars (over 400 house creations) that add depth and complexity to his jus. The resulting lineup of some forty small dishes, a truly herculean task, is surgically crafted and often visually striking: cuttlefish steamed in sake, puffed barley, briny seasoning and seaweed; watercress, smoked beetroot, langoustine milk, iced burnt langoustine heads. Each meal is a feat of culinary excellence, supervised by a diffident, yet single-minded chef, whose watchful eye never leaves the serving hatch."
Addison,"Fairmont Grand Del Mar, 5200 Grand Del Mar Way, San Diego, CA, 92130, USA","San Diego, CA, USA",$$$$,Contemporary,-117.1988908,32.9412972,18583141900,https://guide.michelin.com/en/california/us-san-diego/restaurant/addison,https://www.addisondelmar.com/,3 Stars,0,"Air conditioning,Car park,Garden or park,Interesting wine list,Terrace,Valet parking,Wheelchair access","Global inspiration and Californian sentimentality are at the heart of Chef William Bradley’s world

In [0]:
DESCRIBE TABLE workspace.michelin_analytics.bronze_michelin_restaurants;

col_name,data_type,comment
Name,string,null
Address,string,null
Location,string,null
Price,string,null
Cuisine,string,null
Longitude,double,null
Latitude,double,null
PhoneNumber,bigint,null
Url,string,null
WebsiteUrl,string,null


## STEP 2: Establish the basic size

In [0]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT Name) AS unique_names
FROM workspace.michelin_analytics.bronze_michelin_restaurants;

total_rows,unique_names
19622,18904


## STEP 3: Check the missing values

In [0]:
SELECT
    COUNT(*) AS total_rows,

    SUM(CASE WHEN Name IS NULL THEN 1 ELSE 0 END) AS missing_name,
    SUM(CASE WHEN Address IS NULL THEN 1 ELSE 0 END) AS missing_address,
    SUM(CASE WHEN Location IS NULL THEN 1 ELSE 0 END) AS missing_location,
    SUM(CASE WHEN Price IS NULL THEN 1 ELSE 0 END) AS missing_price,
    SUM(CASE WHEN Cuisine IS NULL THEN 1 ELSE 0 END) AS missing_cuisine,

    SUM(CASE WHEN Longitude IS NULL THEN 1 ELSE 0 END) AS missing_longitude,
    SUM(CASE WHEN Latitude IS NULL THEN 1 ELSE 0 END) AS missing_latitude,

    SUM(CASE WHEN Award IS NULL THEN 1 ELSE 0 END) AS missing_award,
    SUM(CASE WHEN GreenStar IS NULL THEN 1 ELSE 0 END) AS missing_green_star

FROM workspace.michelin_analytics.bronze_michelin_restaurants;

total_rows,missing_name,missing_address,missing_location,missing_price,missing_cuisine,missing_longitude,missing_latitude,missing_award,missing_green_star
19622,0,0,0,0,0,0,0,0,0


### Check duplicate names

In [0]:
SELECT
    Name,
    COUNT(*) AS row_count,
    COUNT(DISTINCT Address) AS unique_addresses,
    COUNT(DISTINCT Location) AS unique_locations
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY Name
HAVING COUNT(*) > 1
ORDER BY row_count DESC, Name;

Name,row_count,unique_addresses,unique_locations
Racines,9,9,9
Fleur de Sel,7,7,7
Krone,6,6,6
Summer Palace,6,6,6
Terra,6,6,6
Adler,5,5,5
Apicius,5,5,5
Benoit,5,5,5
Hirschen,5,5,5
Il Ristorante - Niko Romito,5,5,5


In [0]:
SELECT
    Name,
    Address,
    Location,
    Longitude,
    Latitude,
    COUNT(*) AS row_count
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY
    Name,
    Address,
    Location,
    Longitude,
    Latitude
HAVING COUNT(*) > 1
ORDER BY row_count DESC;

Name,Address,Location,Longitude,Latitude,row_count


### Check blanks

In [0]:
SELECT
    SUM(CASE WHEN TRIM(Name) = '' THEN 1 ELSE 0 END) AS blank_name,
    SUM(CASE WHEN TRIM(Address) = '' THEN 1 ELSE 0 END) AS blank_address,
    SUM(CASE WHEN TRIM(Location) = '' THEN 1 ELSE 0 END) AS blank_location,
    SUM(CASE WHEN TRIM(Price) = '' THEN 1 ELSE 0 END) AS blank_price,
    SUM(CASE WHEN TRIM(Cuisine) = '' THEN 1 ELSE 0 END) AS blank_cuisine,
    SUM(CASE WHEN TRIM(Award) = '' THEN 1 ELSE 0 END) AS blank_award,
    SUM(CASE WHEN TRIM(GreenStar) = '' THEN 1 ELSE 0 END) AS blank_green_star
FROM workspace.michelin_analytics.bronze_michelin_restaurants;

blank_name,blank_address,blank_location,blank_price,blank_cuisine,blank_award,blank_green_star
0,0,0,0,0,0,0


## Categories:

### Award

In [0]:
SELECT
    Award,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY Award
ORDER BY restaurant_count DESC;

Award,restaurant_count
Selected Restaurants,11989
Bib Gourmand,3743
1 Star,3181
2 Stars,547
3 Stars,162


### Green Star

In [0]:
SELECT
    GreenStar,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY GreenStar
ORDER BY restaurant_count DESC;

GreenStar,restaurant_count
0,19045
1,577


## Price


In [0]:
SELECT
    Price,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY Price
ORDER BY restaurant_count DESC;

Price,restaurant_count
$$,7803
$$$,6068
$$$$,4016
$,1676
€€€,22
€€,18
€€€€,10
¥¥¥,4
฿฿,2
€,1


In [0]:
SELECT
    Price,
    Location,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.bronze_michelin_restaurants
WHERE Price NOT IN ('$', '$$', '$$$', '$$$$')
GROUP BY Price, Location
ORDER BY Price, restaurant_count DESC;

Price,Location,restaurant_count
none,"Chon Buri, Thailand",1
¥¥,"Beijing, Chinese Mainland",1
¥¥¥,"Tokyo, Japan",4
฿฿,"Bangkok, Thailand",1
฿฿,"Phuket, Thailand",1
€,"Athens, Greece",1
€€,"Athens, Greece",8
€€,"Milan, Italy",1
€€,"Costigliole d'Asti, Italy",1
€€,"Bressana Bottarone, Italy",1


In [0]:
SELECT *
FROM workspace.michelin_analytics.bronze_michelin_restaurants
WHERE LOWER(TRIM(Price)) = 'none';

Name,Address,Location,Price,Cuisine,Longitude,Latitude,PhoneNumber,Url,WebsiteUrl,Award,GreenStar,FacilitiesAndServices,Description
Apollo Pork Satay,"379 Vachira Prakarn Road, Bang Pla Soi, Mueang Chon Buri, Chon Buri, 20000, Thailand","Chon Buri, Thailand",none,Small eats,100.98575,13.36402,null,https://guide.michelin.com/en/chon-buri-region/chon-buri_2758557/restaurant/apollo-pork-satay,null,Selected Restaurants,0,"Car park,Cash only","Still drawing crowds 70 years on, Apollo Pork Satay is known for its tender pork, liver and intestine skewers – grilled to perfection and served with sweet satay and fresh Acar cucumber. The house-made crab rolls are a crisp, flavoursome bonus, to try with plum or seafood sauce."


## Cuisines

In [0]:
SELECT
    Cuisine,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY Cuisine
ORDER BY restaurant_count DESC
LIMIT 30;

unique_cuisines
267


In [0]:
SELECT
    COUNT(DISTINCT Cuisine) AS unique_cuisines
FROM workspace.michelin_analytics.bronze_michelin_restaurants;

unique_cuisines
267


In [0]:
SELECT
    Cuisine,
    COUNT(*) AS restaurant_count,
    ROUND(
        100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage_of_restaurants
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY Cuisine
ORDER BY restaurant_count DESC
LIMIT 20;

Cuisine,restaurant_count,percentage_of_restaurants
Modern Cuisine,3599,18.34
Contemporary,1088,5.54
Creative,1061,5.41
Traditional Cuisine,934,4.76
Japanese,801,4.08
Italian,671,3.42
Seafood,640,3.26
French,530,2.70
Mediterranean Cuisine,467,2.38
Regional Cuisine,406,2.07


Cross Cuisine

In [0]:
SELECT
    Cuisine,
    COUNT(*) AS total_restaurants,
    SUM(CASE WHEN Award = '1 Star' THEN 1 ELSE 0 END) AS one_star,
    SUM(CASE WHEN Award = '2 Stars' THEN 1 ELSE 0 END) AS two_stars,
    SUM(CASE WHEN Award = '3 Stars' THEN 1 ELSE 0 END) AS three_stars,
    SUM(CASE WHEN Award = 'Bib Gourmand' THEN 1 ELSE 0 END) AS bib_gourmand,
    SUM(CASE WHEN Award = 'Selected Restaurants' THEN 1 ELSE 0 END) AS selected
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY Cuisine
ORDER BY total_restaurants DESC
LIMIT 20;

Cuisine,total_restaurants,one_star,two_stars,three_stars,bib_gourmand,selected
Modern Cuisine,3599,883,88,15,415,2198
Contemporary,1088,220,36,10,113,709
Creative,1061,455,173,63,27,343
Traditional Cuisine,934,19,0,1,210,704
Japanese,801,230,53,14,58,446
Italian,671,59,7,1,114,490
Seafood,640,40,1,4,70,525
French,530,102,13,7,63,345
Mediterranean Cuisine,467,57,4,3,53,350
Regional Cuisine,406,23,0,0,123,260


In [0]:
SELECT
    COUNT(DISTINCT Cuisine) AS unique_cuisine_values,
    COUNT(DISTINCT Location) AS unique_location_values,
    COUNT(DISTINCT Price) AS unique_price_values
FROM workspace.michelin_analytics.bronze_michelin_restaurants;


unique_cuisine_values,unique_location_values,unique_price_values
267,6136,12


## Location

In [0]:
SELECT
    Location,
    COUNT(*) AS restaurant_count
FROM workspace.michelin_analytics.bronze_michelin_restaurants
GROUP BY Location
ORDER BY restaurant_count DESC
LIMIT 50;

Location,restaurant_count
"Tokyo, Japan",545
"Paris, France",451
"London, United Kingdom",375
Singapore,307
"New York, NY, USA",255
"Kyoto, Japan",249
"Osaka, Japan",241
"Hong Kong, Hong Kong SAR China",222
"Bangkok, Thailand",191
"Seoul, South Korea",180


## Coordinates

In [0]:
SELECT
    COUNT(*) AS total_rows,

    SUM(
        CASE
            WHEN Latitude < -90 OR Latitude > 90
            THEN 1 ELSE 0
        END
    ) AS invalid_latitude,

    SUM(
        CASE
            WHEN Longitude < -180 OR Longitude > 180
            THEN 1 ELSE 0
        END
    ) AS invalid_longitude,

    SUM(
        CASE
            WHEN Latitude = 0 AND Longitude = 0
            THEN 1 ELSE 0
        END
    ) AS zero_coordinates

FROM workspace.michelin_analytics.bronze_michelin_restaurants;

total_rows,invalid_latitude,invalid_longitude,zero_coordinates
19622,0,0,0
